# Verification-Before-Assertion — the one-click walkthrough

**What this notebook proves:** a mechanical distinction between *sourced*, *open*, and *invented* claims in model output — reproduced on two **real, recorded falls** from months of human–AI work in Slovene. No API key. No installs. The whole regime is Python standard library.

The three marks:

| Mark | Meaning |
|---|---|
| **⊢** | sourced — carries a citable dictionary source (SSKJ² / Fran, with access date) |
| **⚠** | open — in use, but no dictionary entry reached yet; *the mark is the honesty* |
| **FALL:hybrid** | an invented word caught by lookup alone |

> Unverified is not forbidden. Unmarked is the failure.

In [ ]:
import os
if not os.path.exists("verification-before-assertion"):
    !git clone --depth 1 https://github.com/AtmaOVictu/verification-before-assertion
%cd verification-before-assertion

## 1. The contract

A small, machine-readable lexicon: each word carries its mark, its source, and its gloss. This file (`wordlist-v0.6-demo.json`) is the same artifact an auditor or an eval pipeline would load.

In [ ]:
from vba import load_contract

contract = load_contract("wordlist-v0.6-demo.json")
print(f"{len(contract.lexicon)} lexicon entries, {len(contract.known_hybrids)} known hybrids\n")
for word, e in list(contract.lexicon.items())[:5]:
    print(f"{e.mark} {word:12} {(e.source or '(open)'):8} {e.gloss}")
print("\nKnown hybrid words (invented by models, absent from every dictionary):")
for h in contract.known_hybrids:
    print(f"  x {h}")

## 2. The *lúka* fall — the regime catching its own error

**The recorded real fall (2026-09-30):** the regime's own seed data had sealed the gloss **"opening"** for the word *lúka* with a ⊢ mark. The dictionary says *lúka* = **harbor** ("pristan, zavetišče za ladje"); "opening" belongs to *lúknja* (a hole).

Below: verify the wrong gloss. It must be caught mechanically and recorded — with the scar kept.

In [ ]:
from vba import Register, verify_claim

register = Register()

# The fall: a sealed gloss that contradicts the sourced entry.
verdict = verify_claim("lúka", "opening", contract, register)
print("Wrong gloss:  ", verdict)

# The corrected claim: same word, gloss matching the sourced entry.
verdict_ok = verify_claim("lúka", "pristan, zavetišče za ladje", contract, register)
print("Correct gloss:", verdict_ok)

The register is **append-only**: the fall is not erased by the correction — the correction *carries* the scar. This is the regime's answer to silent post-hoc fixes: a model (or a person) that corrects itself should show the correction, not present the corrected state as if it were always so.

In [ ]:
print(register.export())

## 3. Hybrid words — caught by lookup, no feel required

A **hybrid** is a model-invented compound that passes as real to any reader without a dictionary (e.g. *meritvija* = "measurement + value"). No model, human or artificial, distinguishes these by feel — only by entry. Below, a model output uses one as if it were terminology:

In [ ]:
from vba import mark_claims, report_after_the_act

model_output = ("Model íhta ni merila; meritvija podatka je bila skovanka, "
                "kar je odveč in pusti lúko v dokazu.")

marks = mark_claims(model_output, contract)
for m in marks:
    label = {"⊢": "sourced", "⚠": "open", "FALL:hybrid": "HYBRID CAUGHT"}[m["mark"]]
    print(f"{label:14} {m['word']}")

print("\nReport after the act (L4 - produced from the register, never asserted ahead):")
print(report_after_the_act(register, marks))

## What you just saw

- A sourced claim must match a sourced entry — or it is a **recorded fall** (not a silent fix).
- An open word passes — but **carries the mark** (honesty, not failure).
- A hybrid word is caught **mechanically**, by dictionary lookup alone.

## Honest limits, read before being impressed

- The measured live result (same prompts, with vs. without the contract) is **N=1 model, one run** — reproduce it in `REPRODUCE.md` before generalizing.
- A dictionary is not the world: a genuine neologism is not a hybrid.
- The model can ignore the contract; the measurement is precisely whether it does.

## Where to go deeper

- `QUICKSTART.md` — comprehend, test, understand in ~5 minutes
- `README.md` — the five laws and why they transfer to AI
- `REPRODUCE.md` — re-run the live A/B yourself (Mistral or the Slovenian national model GaMS-9B)
- `PROVENANCE-CHANNEL.md` — the position paper: provenance for contributed language data